## ÉTAPE 12 — Visualisation interactive (Plotly)
Objectif : graphiques interactifs (zoom, survol, filtre)
pip install plotly


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("JEU_DE_DONNEE.csv")


## 1. COURBE INTERACTIVE — ÉVOLUTION DU BLÉ


In [ ]:
df_ble = df[
    (df["category"] == "wheat") &
    (df["country_or_area"] == "World +") &
    (df["element"] == "Production Quantity")
].dropna(subset=["value"]).sort_values("year")
df_ble["valeur_Mt"] = df_ble["value"] / 1e6

fig1 = px.line(
    df_ble, x="year", y="valeur_Mt",
    title="Production mondiale de blé (1961–2007)",
    labels={"year": "Année", "valeur_Mt": "Production (Mt)"},
    markers=True
)
fig1.update_traces(line_color="#2563EB", line_width=2.5)
fig1.write_html("graphique_ble_interactif.html")
print("Graphique 1 sauvegardé : graphique_ble_interactif.html ✓")
fig1.show()


## 2. MULTI-COURBES — MAÏS PAR RÉGION


In [ ]:
regions = ["Northern America +", "Asia +", "Europe +", "South America +", "Africa +"]
couleurs = ["#2563EB", "#DC2626", "#16A34A", "#D97706", "#7C3AED"]

df_maize = df[
    (df["category"] == "maize") &
    (df["element"] == "Production Quantity") &
    (df["country_or_area"].isin(regions))
].dropna(subset=["value"]).copy()
df_maize["valeur_Mt"] = df_maize["value"] / 1e6
df_maize["region"] = df_maize["country_or_area"].str.replace(" +", "", regex=False)

fig2 = px.line(
    df_maize, x="year", y="valeur_Mt", color="region",
    title="Production de maïs par région (1961–2007)",
    labels={"year": "Année", "valeur_Mt": "Production (Mt)", "region": "Région"},
    color_discrete_sequence=couleurs,
    markers=True
)
fig2.write_html("maize_regions_interactif.html")
print("Graphique 2 sauvegardé : maize_regions_interactif.html ✓")
fig2.show()


## 3. BARRES INTERACTIVES — TOP 10 CULTURES


In [ ]:
df_prod = df[
    (df["country_or_area"] == "World +") &
    (df["element"] == "Production Quantity")
].dropna(subset=["value"])

top10 = (
    df_prod.groupby("category")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10.columns = ["Culture", "Production"]
top10["Production_Gt"] = top10["Production"] / 1e9

fig3 = px.bar(
    top10.sort_values("Production_Gt"),
    x="Production_Gt", y="Culture",
    orientation="h",
    title="Top 10 des cultures les plus produites (1961–2007)",
    labels={"Production_Gt": "Production totale (milliards tonnes)", "Culture": ""},
    color="Production_Gt",
    color_continuous_scale="Blues"
)
fig3.write_html("top10_interactif.html")
print("Graphique 3 sauvegardé : top10_interactif.html ✓")
fig3.show()


## 4. SCATTER INTERACTIF — SURFACE vs RENDEMENT


In [ ]:
df_pivot = df[
    (df["country_or_area"] == "World +") &
    (df["element"].isin(["Area Harvested", "Yield"]))
].dropna(subset=["value"])

tableau = df_pivot.pivot_table(
    index=["category", "year"],
    columns="element", values="value", aggfunc="sum"
).reset_index().dropna()

tableau.columns = ["category", "year", "surface_ha", "rendement"]
tableau["surface_Mha"] = tableau["surface_ha"] / 1e6
tableau["rendement_t"] = tableau["rendement"] / 1e4

fig4 = px.scatter(
    tableau, x="surface_Mha", y="rendement_t",
    color="category", hover_data=["year"],
    title="Surface cultivée vs Rendement (toutes cultures, 1961–2007)",
    labels={
        "surface_Mha": "Surface (millions Ha)",
        "rendement_t": "Rendement (t/Ha)",
        "category": "Culture"
    },
    opacity=0.6
)
fig4.write_html("scatter_surface_rendement.html")
print("Graphique 4 sauvegardé : scatter_surface_rendement.html ✓")
fig4.show()

print("\n=== RÉSUMÉ PLOTLY ===")
print("px.line()    → courbe interactive")
print("px.bar()     → barres interactives")
print("px.scatter() → nuage de points interactif")
print("fig.show()   → ouvrir dans le navigateur")
print("fig.write_html('nom.html') → sauvegarder")
